# 🖼️ Serveur Backend Stable Diffusion sur Google Colab — NeuroChat

Ce notebook Jupyter permet d'exécuter un serveur de génération d'images basé sur **FastAPI** et **Stable Diffusion** en exploitant la puissance du processeur graphique (GPU) gratuit proposé par Google Colab.

Le serveur local ainsi démarré est ensuite exposé de manière publique et sécurisée sur internet via le service **Localtunnel**, vous permettant de l'interroger directement depuis l'interface web **NeuroChat** à l'aide de la commande `/image` ou du bouton ✨.

---

## 🔒 Sécurité et gestion des jetons (Token Hugging Face)

Pour télécharger le modèle Stable Diffusion, un compte Hugging Face et un jeton d'accès (token) sont requis.

> **⚠️ Recommandation de sécurité importante :**  
> Ne codez jamais vos clés API ou jetons d'accès en clair dans vos notebooks si vous prévoyez de les publier sur GitHub.
>
> **Comment configurer le jeton de manière sécurisée dans Colab ?**
> 1. Créez un compte gratuit sur [huggingface.co](https://huggingface.co) et générez un jeton dans *Settings -> Access Tokens*.
> 2. Dans Google Colab, cliquez sur l'icône de clé 🔑 dans la barre latérale gauche (onglets *Secrets*).
> 3. Ajoutez un nouveau secret ayant pour nom `HF_TOKEN` et collez-y votre jeton Hugging Face.
> 4. Activez l'accès au notebook pour ce secret.

### ⚡ Étape 1 — Installation des dépendances

Nous installons les packages nécessaires pour exécuter le modèle de diffusion, lancer le serveur web asynchrone (FastAPI + Uvicorn) et installer Localtunnel.

In [ ]:
# Installation des paquets Python requis
!pip install fastapi uvicorn diffusers transformers accelerate pydantic nest-asyncio

# Installation globale de Localtunnel via le gestionnaire de paquets Node.js (npm)
!npm install -g localtunnel

### 🧠 Étape 2 — Démarrage du serveur et création du tunnel

Le code ci-dessous charge le modèle Stable Diffusion v1.5 sur la carte graphique GPU (si disponible) et configure l'API FastAPI :
- `POST /generate` : Reçoit une description textuelle (prompt) et génère l'image correspondante renvoyée au format brut PNG.

In [ ]:
import os
import io
import torch
import nest_asyncio
import uvicorn
import subprocess
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
from diffusers import StableDiffusionPipeline

# ── Récupération sécurisée du Token Hugging Face ───────────────────────────
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ["HF_TOKEN"] = token
        print("[OK] Jeton HF_TOKEN charge depuis les secrets Google Colab.")
    else:
        raise ValueError("Le secret HF_TOKEN est vide.")
except Exception as e:
    # Jeton par défaut de repli en cas d'absence de configuration de secret
    print(f"[INFO] Impossible de lire le secret Colab : {e}")
    print("[REPLI] Utilisation du jeton Hugging Face de secours...")
    os.environ["HF_TOKEN"] = "VOTRE_TOKEN_ICI"

# Initialisation de l'application FastAPI
app = FastAPI(title="Serveur Stable Diffusion pour NeuroChat")

# Configuration du CORS (Cross-Origin Resource Sharing)
# Permet à notre frontend local (http://localhost:8000) d'appeler ce serveur sans blocage de sécurité
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Structure de la requête entrante
class GenerateRequest(BaseModel):
    prompt: str

# Variable globale pour stocker le modèle
pipe = None
MODEL_ID = "runwayml/stable-diffusion-v1-5"

# Chargement asynchrone du modèle au démarrage
@app.on_event("startup")
async def load_model():
    global pipe
    print(f"Chargement du modele : {MODEL_ID}...")
    
    # Sélection automatique du périphérique de calcul GPU (cuda) ou CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    
    print(f"Calcul execute sur : {device.upper()}")
    
    # Téléchargement et chargement des poids du réseau neuronal
    pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
    pipe = pipe.to(device)
    print("Le modele de generation a ete charge avec succes !")

# Endpoint principal de génération
@app.post("/generate")
async def generate_image(req: GenerateRequest):
    if pipe is None:
        raise HTTPException(status_code=503, detail="Modele non disponible ou en cours de chargement")
    
    try:
        # Lancement du processus de diffusion d'image
        print(f"Generation pour le prompt : '{req.prompt}'")
        image = pipe(req.prompt).images[0]
        
        # Encodage de l'image résultante au format binaire PNG
        img_byte_arr = io.BytesIO()
        image.save(img_byte_arr, format='PNG')
        
        # Retour de l'image brute avec le type mime approprié
        return Response(content=img_byte_arr.getvalue(), media_type="image/png")
        
    except Exception as e:
        print(f"[ERREUR] Echec lors de la generation : {e}")
        raise HTTPException(status_code=500, detail=str(e))

# ── Exposition publique via Localtunnel ───────────────────────────────────
print("Demarrage du tunnel public Localtunnel...")
lt_process = subprocess.Popen(['lt', '--port', '8000'], stdout=subprocess.PIPE)
time.sleep(2.5)

# Lecture de l'URL publique générée par localtunnel
url_output = lt_process.stdout.readline().decode('utf-8').strip()
public_url = url_output.replace('your url is: ', '')

print("="*70)
print(f"⭐ COPIEZ CETTE URL DANS LES PARAMETRES NEUROCHAT (BARRE LATERALE) :")
print(f"👉 {public_url} 👈")
print("="*70)

# Application du patch de boucle asynchrone (nécessaire dans les notebooks)
nest_asyncio.apply()

# Démarrage du serveur Uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)
